In [ ]:
%pip install anthropic python-dotenv

: 

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [6]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-5-5"

In [ ]:
CONTENT_KEY, SYSTEM_KEY = "content", "system"
USER_KEY, ASSISTANT_KEY = "user", "assistant"

class Chat:
    def __init__(self, client: Anthropic, system: str=None):
        self.client = client
        self.message_store = []
        self.system = system

    def get_text(self, message) -> str:
        return next(block.text for block in message.content if block.type == "text")

    def send_message(self, message) -> str:
        self.message_store.append({
            "role": USER_KEY,
            "content": message
        })

        args = {
            "model": model,
            "max_tokens": 1000,
            "messages": self.message_store,
        }
        if self.system:
            args[SYSTEM_KEY] = self.system
                
        message = self.client.messages.create(**args)

        self.message_store.append({
            "role": ASSISTANT_KEY,
            "content": self.get_text(message)
        })

        return self.message_store[-1][CONTENT_KEY]


system_prompt = """
You are a competitive programming tutor. Your goal is to help the student get better at solving problems on their own, not to solve problems for them.

When the student shares a problem:
- First make sure they understand it. Ask them to restate it or work through a small example if they seem unsure.
- Draw attention to the constraints and what they imply about the required time complexity (for example, n up to 10^5 usually rules out O(n^2)).
- Guide with questions and hints rather than answers. Start with a gentle nudge and only give a more direct hint if they are still stuck after trying.
- Do not write a full solution unless the student explicitly asks for one. When they do, explain the idea first, then the code.

When the student shares their code:
- Check correctness first, then complexity, then style.
- If it fails, help them find the bug themselves: suggest a test case that breaks it rather than pointing at the line.
- Point out edge cases they may have missed: empty input, n = 1, duplicates, negative numbers, overflow, off-by-one errors.

When explaining concepts:
- Name the underlying technique (two pointers, prefix sums, binary search on the answer, DP, BFS/DFS, greedy, etc.) so the student learns to recognize patterns across problems.
- State time and space complexity for every approach you discuss.
- Keep explanations concise and use small concrete examples.
- Include time and space complexity in big O notation
- Explain worst case and best case scenarios

Use Python for code unless the student uses another language. Be encouraging but honest: if an approach will not pass the constraints, say so clearly.
"""
ch = Chat(client)

while True:
    user_prompt = input("> ")
    print(f"request: {user_prompt}")
    if user_prompt == "!q": break

    response = ch.send_message(user_prompt)
    print(f"response: {response}")

request: help me solve two sum
response: The **Two Sum** problem asks: given an array `nums` and an integer `target`, return the indices of two numbers that add up to `target`. Each input has exactly one solution, and you can't use the same element twice.

**Example**
```
nums = [2, 7, 11, 15], target = 9
Output: [0, 1]   because nums[0] + nums[1] = 2 + 7 = 9
```

## Approach 1: Brute Force (O(n²))

Check every pair of numbers.

```python
def two_sum(nums, target):
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] + nums[j] == target:
                return [i, j]
```

This works but is slow for large inputs.

## Approach 2: Hash Map (O(n) time, O(n) space)

For each number `x`, the number we need is `target - x`. If we've already seen that complement, we're done. Otherwise, store `x` with its index and keep going.

```python
def two_sum(nums, target):
    seen = {}  # maps value -> index
    for i, num in enumerate(nums):
        complemen